In [ ]:
%load_ext bigquery_magics

import bigquery_magics
bigquery_magics.context.project = "k-move1-kyungpil"

BigQueryは、一般的なRDBMSでは扱いにくい`ネストされたデータ（Nested Data）`をサポートしている。

主なデータ型は次のとおりである。

- `ARRAY`：配列（例：Pythonの`['東京', '京都', '大阪']`型）
- `STRUCT`：オブジェクト・レコード（例：Pythonの`{'name': '千尋', 'city': '東京'}`型）
- `ARRAY<STRUCT>`：オブジェクトの配列（例：Pythonの`[{}, {}, ...]`型）

実務では、単純な配列よりも`ARRAY<STRUCT>`形式がよく使用される。

## 1. ARRAY

一つの列に複数の値を配列として保存できる。

In [ ]:
%%bigquery

WITH tmp AS (
    SELECT
        ['読書', '東京散策', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobbies,
    name
FROM tmp;

| 行 | hobby | name |
|---:|---|---|
| 1 | サッカー<br>映画<br>ゲーム | 千尋 |

#### # 特定の要素を取得

In [5]:
%%bigquery --project k-move1-kyungpil

WITH tmp AS (
    SELECT
        ['読書', '東京散策', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobbies[0] AS first_hobby, -- hobby[0]は hobby[OFFSET(0)] の形に変更されて実行
    name
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,first_hobby,name
0,読書,千尋


#### # 数値 ARRAY(配列)

In [8]:
%%bigquery

WITH tmp AS (
    SELECT [10, 20, 30, 40] AS scores
)

SELECT scores
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,scores
0,"[10, 20, 30, 40]"


## 2. ARRAY型でよく使用する機能

#### 1）UNNESTで配列を複数行に展開する

`UNNEST`は、配列を受け取り、各要素を複数の行に展開してテーブルとして返す。

In [11]:
%%bigquery

WITH tmp AS (
    SELECT
        ['サッカー', '映画', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobby,
    name
FROM tmp
CROSS JOIN UNNEST(hobbies) AS hobby;

-- `hobbies`配列の各要素を`hobby`という行に分離する。
-- `tmp`と`UNNEST(hobbies)`を`CROSS JOIN`するため、3行が返される。

Query is running:   0%|          |

Downloading:   0%|          |

,hobby,name
0,サッカー,千尋
1,映画,千尋
2,ゲーム,千尋


#### # カンマを使用した省略形

In [12]:
%%bigquery

WITH tmp AS (
    SELECT
        ['サッカー', '映画', 'ゲーム'] AS hobbies,
        '千尋' AS name
)

SELECT
    hobby,
    name
FROM tmp, UNNEST(hobbies) AS hobby;

Query is running:   0%|          |

Downloading:   0%|          |

,hobby,name
0,サッカー,千尋
1,映画,千尋
2,ゲーム,千尋


次の二つは同じ意味である。

```sql
FROM tmp
CROSS JOIN UNNEST(hobbies) AS hobby;
```

```sql
FROM tmp, UNNEST(hobbies) AS hobby;
```

> BigQueryはカンマを使用した結合もサポートするが、実務では可読性の高いANSI JOIN構文(JOIN ... ON )を使用することが望ましい。

    #### # カンマ結合とANSI JOINの比較

次の二つのクエリは同じ意味である。

```sql
SELECT *
FROM a, b
WHERE a.id = b.id;

SELECT *
FROM a
INNER JOIN b
    ON a.id = b.id;
```

次の二つも同じ結果になる。

```sql
SELECT *
FROM a, b;

SELECT *
FROM a
CROSS JOIN b;
```

#### # 1) 実習

In [15]:
%%bigquery

WITH customer AS (
    SELECT 1 AS customer_id, '千尋' AS customer_name UNION ALL
    SELECT 2, 'ハウル' UNION ALL
    SELECT 3, 'キキ'
),
orders AS (
    SELECT 101 AS order_id, 1 AS customer_id, 'プリウス' AS product UNION ALL
    SELECT 102, 1, 'スープラ' UNION ALL
    SELECT 103, 3, 'シビック'
)

-- INNER JOIN
select * 
FROM orders o, customer c
where o.customer_id = c.customer_id; 

-- CROSS JOIN
-- select * 
-- FROM orders o, customer c;

Query is running:   0%|          |

Downloading:   0%|          |

,order_id,customer_id,product,customer_id_1,customer_name
0,101,1,プリウス,1,千尋
1,102,1,スープラ,1,千尋
2,103,3,シビック,3,キキ


#### # 2) 実習

In [16]:
%%bigquery

WITH customer AS (
    SELECT 1 AS customer_id, '千尋' AS customer_name UNION ALL
    SELECT 2, 'ハウル' UNION ALL
    SELECT 3, 'キキ'
),
orders AS (
    SELECT 101 AS order_id, 1 AS customer_id, 'プリウス' AS product UNION ALL
    SELECT 102, 1, 'スープラ' UNION ALL
    SELECT 103, 3, 'シビック'
)

-- CROSS JOIN
select * 
FROM orders o, customer c;

Query is running:   0%|          |

Downloading:   0%|          |

,order_id,customer_id,product,customer_id_1,customer_name
0,101,1,プリウス,1,千尋
1,101,1,プリウス,2,ハウル
2,101,1,プリウス,3,キキ
3,102,1,スープラ,1,千尋
4,102,1,スープラ,2,ハウル
5,102,1,スープラ,3,キキ
6,103,3,シビック,1,千尋
7,103,3,シビック,2,ハウル
8,103,3,シビック,3,キキ


#### 2）数値ARRAYの単一行関数

| 関数 | 説明 |
|---|---|
| `ARRAY_LENGTH()` | 配列の要素数 |
| `ARRAY_FIRST()` | 最初の要素 |
| `ARRAY_LAST()` | 最後の要素 |
| `OFFSET()` | 0から始まるインデックス |
| `ORDINAL()` | 1から始まるインデックス |
| `SAFE_OFFSET()` | 範囲外の場合は`NULL` |
| `ARRAY_REVERSE()` | 配列の順序を反転 |
| `ARRAY_SLICE()` | 配列の一部を抽出（開始位置・終了位置を含む） |
| `ARRAY_TO_STRING()` | 配列の要素を一つの文字列に連結 |

In [18]:
%%bigquery

WITH tmp AS (
SELECT
    [10,20,30,40] AS scores
)

SELECT
    scores,
    ARRAY_LENGTH(scores) AS length,
    ARRAY_FIRST(scores) AS first_value,
    ARRAY_LAST(scores) AS last_value,
    scores[OFFSET(0)] AS offset0,
    scores[OFFSET(2)] AS offset2,
    scores[ORDINAL(1)] AS ordinal1,
    scores[ORDINAL(3)] AS ordinal3,
    scores[SAFE_OFFSET(10)] AS safe_offset,
    ARRAY_REVERSE(scores) AS reverse,
    ARRAY_SLICE(scores, 1, 3) AS slice_1_3,
    ARRAY_TO_STRING(
        ARRAY(
            SELECT CAST(x AS STRING)
            FROM UNNEST(scores) x
        ),
        ', '
    ) AS to_string
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,scores,length,first_value,last_value,offset0,offset2,ordinal1,ordinal3,safe_offset,reverse,slice_1_3,to_string
0,"[10, 20, 30, 40]",4,10,40,10,30,10,30,<NA>,"[40, 30, 20, 10]","[20, 30, 40]","10, 20, 30, 40"


#### 3）文字列ARRAYの単一行関数

In [19]:
%%bigquery

WITH tmp AS ( SELECT
    [
    'SQL',
    'Python',
    'BigQuery',
    'Power BI'
    ] AS skills
)

SELECT
    skills,
    ARRAY_LENGTH(skills) AS length,
    ARRAY_FIRST(skills) AS first_skill,
    ARRAY_LAST(skills) AS last_skill,
    skills[OFFSET(1)] AS second,
    skills[ORDINAL(3)] AS third,
    ARRAY_REVERSE(skills) AS reverse,
    ARRAY_SLICE(skills,1,3) AS slice,
    ARRAY_TO_STRING(skills,' | ') AS string_list,
    skills[SAFE_OFFSET(10)] AS safe_value
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,skills,length,first_skill,last_skill,second,third,reverse,slice,string_list,safe_value
0,"[SQL, Python, BigQuery, Power BI]",4,SQL,Power BI,Python,BigQuery,"[Power BI, BigQuery, Python, SQL]","[Python, BigQuery, Power BI]",SQL | Python | BigQuery | Power BI,NaN


#### 4）配列を結合する関数

`ARRAY_CONCAT()`は、複数の配列を一つに結合する。

In [20]:
%%bigquery

SELECT ARRAY_CONCAT([10,20],[30,40],[50]) AS result;

Query is running:   0%|          |

Downloading:   0%|          |

,result
0,"[10, 20, 30, 40, 50]"


#### 5）ARRAY生成関数

配列を新しく生成する関数もよく使用される。

| 関数 | 説明 |
|---|---|
| `GENERATE_ARRAY()` | 数値配列を生成 |
| `GENERATE_DATE_ARRAY()` | 日付配列を生成 |
| `GENERATE_TIMESTAMP_ARRAY()` | タイムスタンプ配列を生成 |

In [21]:
%%bigquery

SELECT
    GENERATE_ARRAY(1, 10) AS `1から10まで`,
    GENERATE_ARRAY(0, 100, 20) AS `0から100まで20間隔`,
    GENERATE_DATE_ARRAY(
        DATE '2026-07-01',
        DATE '2026-07-07'
    ) AS `7月1日から7月7日まで`;

Query is running:   0%|          |

Downloading:   0%|          |

,1から10まで,0から100まで20間隔,7月1日から7月7日まで
0,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0, 20, 40, 60, 80, 100]","[2026-07-01T00:00:00.000, 2026-07-02T00:00:00...."


#### 4）ARRAY集約関数（Aggregate）

| 関数 | 説明 |
|---|---|
| `ARRAY_AGG()` | 複数の行を一つの配列にまとめる |
| `ARRAY_CONCAT_AGG()` | 複数の配列を一つの配列に結合する |

In [22]:
%%bigquery

WITH tmp AS (
    SELECT 'SQL' AS subject UNION ALL
    SELECT 'Python' UNION ALL
    SELECT 'BigQuery' UNION ALL
    SELECT 'SQL'
)  -- 文字列4行のテーブル（ARRAY型ではない）

SELECT
    ARRAY_AGG(subject) AS subjects  -- ARRAY型に集約
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,subjects
0,"[SQL, Python, BigQuery, SQL]"


In [23]:
%%bigquery

WITH tmp AS (
  SELECT 'SQL' subject UNION ALL
  SELECT 'Python' UNION ALL
  SELECT 'BigQuery' UNION ALL
  SELECT 'SQL'
)
SELECT
  ARRAY_AGG(subject ORDER BY subject) AS sorted,
  ARRAY_AGG(DISTINCT subject) AS distinct_subjects
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,sorted,distinct_subjects
0,"[BigQuery, Python, SQL, SQL]","[SQL, Python, BigQuery]"


In [24]:
%%bigquery

WITH tmp AS (
  SELECT [1,2] arr UNION ALL
  SELECT [3,4] UNION ALL
  SELECT [5]
)

SELECT
  ARRAY_CONCAT_AGG(arr) AS result
FROM tmp;

Query is running:   0%|          |

Downloading:   0%|          |

,result
0,"[1, 2, 3, 4, 5]"
